In [1]:
import numpy as np
import cv2

np.random.seed(42)
cv2.setRNGSeed(42)
PATH_RUNNERS = '/home/gabrielcarmonapy/Desktop/camera-vision/videos/runners.mp4'
PATH_BRIDGES = '/home/gabrielcarmonapy/Desktop/camera-vision/videos/brigde.mp4'
PATH_FISHES = '/home/gabrielcarmonapy/Desktop/camera-vision/videos/fish.mp4'
PATH_STREET_1 = '/home/gabrielcarmonapy/Desktop/camera-vision/videos/street.mp4'
PATH_STREET_2 = '/home/gabrielcarmonapy/Desktop/camera-vision/videos/street_.mp4'
PATH_FRENCH = '/home/gabrielcarmonapy/Desktop/camera-vision/videos/french.mp4'

cap = cv2.VideoCapture(PATH_RUNNERS)

In [2]:
def np_stack_logic() -> np.ndarray:
  # stack order => (N,H,W,C)

  frame_1 = np.random.randint(255,size=(100,100,3)) # (H,W,C)
  frame_2 = np.random.randint(255,size=(100,100,3))

  frames = np.hstack([frame_1,frame_2])

  print(frames.shape)
  print(frames)

In [3]:
# setting the background image to a gray scale
from typing import List

def frameList(path_video,quantity_frames:int) -> List[np.ndarray]:
  ''''''
  # list of frames init
  frames = []

  # video capturing from path_video
  cap = cv2.VideoCapture(path_video)

  # total frames
  total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

  # random frames
  frames_selected_randomly = np.random.randint(total_frames,size=quantity_frames)

  # loop topick the frames id and append them into a list
  for ids in frames_selected_randomly :
    cap.set(cv2.CAP_PROP_POS_FRAMES,ids)
    has_frame,frame = cap.read()

    if has_frame:

       frames.append(frame)

  cap.release()
  return frames


def backgroundImage(frame_list) -> np.ndarray:
  # stack order => (N,H,W,C)
  stacked = np.stack(frame_list)
   # median per pixel
  median_frames = np.median(frame_list,axis=0).astype(np.uint8)
  return median_frames

def backgroundGrayScale(median_frame_list) -> np.ndarray:
  # Convert the image to GRAY
   gray = cv2.cvtColor(median_frame_list,cv2.COLOR_BGR2GRAY)
   return gray


In [4]:
frame_list = frameList(path_video=PATH_RUNNERS, quantity_frames=12)
median_image = backgroundImage(frame_list=frame_list)
gray_image = backgroundGrayScale(median_frame_list=median_image)

def saveImage(name_of_the_image, image):
    frame_width = 680
    frame_height = 480 
    
    resized = cv2.resize(image, (frame_width, frame_height))
    cv2.imwrite(name_of_the_image, resized)

saveImage('median_background.png',image=median_image)
saveImage('median_gray_background.png',image=gray_image)

In [5]:
def allGrayImages(path_video,video:bool):
  while video:
    cap = cv2.VideoCapture(path_video)
    has_frame,frame = cap.read()

    if not has_frame:
      raise('Any images left')
      break

    gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)

    cv2.imshow(gray)
    if cv2.waitKey(1) & 0xFF == ord('q'):
      break

  cap.release()

def allColorImages(path_video,video:bool):
  while video :
    cap = cv2.VideoCapture(path_video)
    has_frame,frame = cap.read()

    if not has_frame:
      raise('Any images left')
      break

    frame_gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    difference_frame = cv2.absdiff(frame_gray)

    th,dframe = cv2.threshold(difference_frame,30,255,cv2.THRESH_BINARY) # black or white

    cv2.imshow(difference_frame)

In [6]:
algorithm_types = ['KNN','MOG','MOG2']
algorithm_selected = algorithm_types[1] # MOG == best one

def applyingMask(type_of_mask):
  
  algorithm_types = {
      'KNN': cv2.createBackgroundSubtractorKNN,
      'MOG': cv2.bgsegm.createBackgroundSubtractorMOG,
      'MOG2': cv2.createBackgroundSubtractorMOG2
  }
  if type_of_mask not in algorithm_types:
    raise ValueError("Mask not in the array")
  
  return algorithm_types[type_of_mask]()

In [7]:
# === Kernel ===
def get_kernel(kernel_type):
    kernel_algorithm = {
        'dilation': cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)),
        'ones': np.ones((3, 3), np.uint8),
    }

    if kernel_type not in kernel_algorithm:
        raise ValueError("Wrong type. Choose between: dilation, ones")

    return kernel_algorithm[kernel_type]


def filterWithKernel(image: np.ndarray, filter: str):

    if filter == 'closing':
        return cv2.morphologyEx(image, cv2.MORPH_CLOSE, get_kernel('ones'), iterations=2)

    elif filter == 'opening':
        return cv2.morphologyEx(image, cv2.MORPH_OPEN, get_kernel('ones'), iterations=2)

    elif filter == 'dilation':
        return cv2.dilate(image, get_kernel('dilation'), iterations=2)

    elif filter == 'combine':
        closing = cv2.morphologyEx(image, cv2.MORPH_CLOSE, get_kernel('ones'), iterations=2)
        opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN, get_kernel('ones'), iterations=2)
        dilation = cv2.dilate(opening, get_kernel('dilation'), iterations=2)
        return dilation

    else:
        raise ValueError("Choose between: opening, closing, dilation or combine")
   

In [8]:
print('Type of kernels:')
print('*' * 25)
print(get_kernel('dilation'))
print('*' * 25)
print(get_kernel('ones'))

Type of kernels:
*************************
[[0 1 0]
 [1 1 1]
 [0 1 0]]
*************************
[[1 1 1]
 [1 1 1]
 [1 1 1]]


w_min = 30 - which corresponds to the minimum width of the rectangle we will use around the car;

h_min = 30 - refers to the minimum height;
offset = 10 - is the allowed error value between pixels after passing through the ROI count line;
line_ROI = 500 - ROI line or region of interest line is the position of the count line in the video;
people = 0 - the initial value of cars.

cv2.putText(img, text, org, fontFace, fontScale, color, thickness, lineType, bottomLeftOrigin)

In [ ]:
def get_centroid(x, y, w, h):
    cx = x + int(w / 2)
    cy = y + int(h / 2)
    return cx, cy

def process_detection(detec, obj, frame, line_y, offset):
    for (x, y) in detec:
        if (line_y + offset) > y > (line_y - offset):
            obj += 1
            cv2.line(frame, (0, line_y), (1200, line_y), (0, 255, 0), 3)
            detec.remove((x, y))
            print(f"Object detected! Total: {obj}")
    return obj

def runningImagesWith(path_video):

    w_min, h_min = 30, 30
    offset = 6
    line_ROI = 400 
    obj_count = 0
    detec = []

    cap = cv2.VideoCapture(path_video)
    # Background Subtractor (Assuming 'applyingMask' was a wrapper for this)
    subtractor = cv2.createBackgroundSubtractorMOG2()
    fps = cap.get(cv2.CAP_PROP_FPS)

    while cap.isOpened():
        has_frame, frame = cap.read()
        if not has_frame: break

        frame = cv2.resize(frame, (680, 480))
        mask = subtractor.apply(frame)
        
        # Simple cleaning (replaces your filterWithKernel)
        _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Draw the counting line
        cv2.line(frame, (0, line_ROI), (680, line_ROI), (255, 127, 0), 2)

        for c in contours:
            (x, y, w, h) = cv2.boundingRect(c)
            if (w >= w_min) and (h >= h_min):
                # 1. Draw Bounding Box
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
                
                # 2. Get and Draw Centroid
                center = get_centroid(x, y, w, h)
                detec.append(center)
                cv2.circle(frame, center, 4, (0, 0, 255), -1)

        # 3. Update Count
        obj_count = process_detection(detec, obj_count, frame, line_ROI, offset)

        # 4. Display Info
        #cv2.putText(frame, f"COUNT: {obj_count}", (10, 50), 
        #           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
        cv2.imshow('Video Analysis', frame)
        
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [10]:
mask_algorithm = applyingMask(algorithm_selected)
print(f"Selected algorithm: {algorithm_selected}")
print(f"Created mask object: {mask_algorithm}")

Selected algorithm: MOG
Created mask object: < cv2.bgsegm.BackgroundSubtractorMOG 0x718decbbedd0>


In [ ]:
def main():
   try :
   # Init variables
      runningImagesWith(PATH_BRIDGES)
   except Exception as e:
      raise ValueError(f"Something went wrong : {e}")

main()

QFontDatabase: Cannot find font directory /home/gabrielcarmonapy/Desktop/camera-vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/gabrielcarmonapy/Desktop/camera-vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/gabrielcarmonapy/Desktop/camera-vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/gabrielcarmonapy/Desktop/camera-vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ fo

Object detected! Total: 1
Object detected! Total: 2
Object detected! Total: 3
Object detected! Total: 4
Object detected! Total: 5
Object detected! Total: 6
Object detected! Total: 7
Object detected! Total: 8
Object detected! Total: 9
Object detected! Total: 10
Object detected! Total: 11
Object detected! Total: 12
Object detected! Total: 13
Object detected! Total: 14
Object detected! Total: 15
Object detected! Total: 16
Object detected! Total: 17
Object detected! Total: 18
Object detected! Total: 19
Object detected! Total: 20
Object detected! Total: 21
Object detected! Total: 22
Object detected! Total: 23
Object detected! Total: 24
Object detected! Total: 25
Object detected! Total: 26
Object detected! Total: 27
Object detected! Total: 28
Object detected! Total: 29
Object detected! Total: 30
Object detected! Total: 31
Object detected! Total: 32
Object detected! Total: 33
Object detected! Total: 34
Object detected! Total: 35
Object detected! Total: 36
Object detected! Total: 37
Object det

In [ ]:
cap_testing = cv2.VideoCapture("/home/gabrielcarmonapy/Desktop/camera-vision/mask_filters.avi")

print("Opened:", cap_testing.isOpened())
print("Frames:", cap_testing.get(cv2.CAP_PROP_FRAME_COUNT))

Opened: False
Frames: 0.0


In [ ]:
# functions activation
# frames = frameList(PATH_RUNNERS,12)
# median_frames = backgroundImage(frames)
# gray_median_frame = backgroundGrayScale(median_frames)

In [ ]:
# cv2.imshow(median_frames)

In [ ]:
# cv2.imshow(gray_median_frame)